In [ ]:
# Parameters
precision_qubits = 4
phase_theta = 0.25


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace
import matplotlib.pyplot as plt
import numpy as np

n = int(precision_qubits)
phase = float(phase_theta)

print(f"Quantum Phase Estimation — {n} precision qubits, θ = {phase}")

total = n + 1
qc = QuantumCircuit(total, n)

# Prepare eigenstate |1⟩ on target qubit
qc.x(n)

# Hadamard on counting register
for i in range(n):
    qc.h(i)
qc.barrier()

# Controlled-U^(2^k) operations
for k in range(n):
    angle = 2 * np.pi * phase * (2**k)
    qc.cp(angle, k, n)
qc.barrier()

# Inverse QFT on counting register
for j in range(n // 2):
    qc.swap(j, n - j - 1)
for j in range(n):
    for k in range(j):
        qc.cp(-np.pi / (2**(j-k)), k, j)
    qc.h(j)

# Measure counting register
for i in range(n):
    qc.measure(i, i)

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()

# Find most likely result
max_state = max(counts, key=counts.get)
estimated_phase = int(max_state, 2) / (2**n)
print(f"Most likely measurement: |{max_state}⟩ ({counts[max_state]} shots)")
print(f"Estimated phase: {estimated_phase:.4f} (actual: {phase})")
print(f"All counts: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere for qubit 0
qc_sv = QuantumCircuit(total)
qc_sv.x(n)
for i in range(n):
    qc_sv.h(i)
for k in range(n):
    angle = 2 * np.pi * phase * (2**k)
    qc_sv.cp(angle, k, n)
for j in range(n // 2):
    qc_sv.swap(j, n - j - 1)
for j in range(n):
    for k in range(j):
        qc_sv.cp(-np.pi / (2**(j-k)), k, j)
    qc_sv.h(j)
sv = Statevector.from_instruction(qc_sv)
rho = partial_trace(sv, list(range(1, total)))
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi_bloch = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi_bloch:.6f}")
